In [1]:
# Install Klygo from the v2-release branch
!pip install --upgrade --no-cache-dir git+https://github.com/IchigoMazone/klygo.git@v2-release

# Mining Duplicates

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/IchigoMazone/klygo/blob/v2-release/tutorials/mining/Document_Mining_Duplicates.ipynb)

Identify and filter exact and perceptual duplicate images across datasets.

## Jump to an API

- [mining.find_duplicates](#mining.find_duplicates)
- [mining.deduplicate](#mining.deduplicate)
- [mining.DuplicateReport](#mining.DuplicateReport)

### mining.find_duplicates

Report exact and perceptual duplicate groups without deleting data.

In [2]:
from klygo import archive, datasets, files, mining

workspace = files.join("artifacts", "tutorials", "mining")
flat_dir = files.join(workspace, "flat_coco8")
if not files.exists(files.join(flat_dir, "data.yaml")):
    files.mkdir(workspace)
    zip_path = files.download("https://github.com/ultralytics/assets/releases/download/v0.0.0/coco8.zip", workspace, overwrite=True, verbose=False)
    yaml_path = files.download("https://raw.githubusercontent.com/ultralytics/ultralytics/main/ultralytics/cfg/datasets/coco8.yaml", workspace, overwrite=True, verbose=False)
    raw_dir = files.join(workspace, "raw_coco8")
    archive.extract(zip_path, raw_dir, overwrite=True, verbose=False)
    files.copy(yaml_path, files.join(raw_dir, "coco8.yaml"), overwrite=True)
    datasets.convert(
        source=raw_dir,
        target=flat_dir,
        tree={
            "images": ["coco8/images/train", "coco8/images/val"],
            "labels": ["coco8/labels/train", "coco8/labels/val"],
            "metadata": "coco8.yaml",
        },
        on_conflict="rename",
        overwrite=True,
    )

data = datasets.load(flat_dir, format="yolo", verbose=False)
print(f"Loaded dataset: {len(data)} images, {data.total_objects} boxes, {len(data.names)} classes")

duplicates = mining.find_duplicates(data, exact=True, method="phash", threshold=0.95)
duplicates.show()
print(f"Detected {len(duplicates.groups)} duplicate groups, {duplicates.duplicate_images} removable images.")

Loaded dataset: 8 images, 30 boxes, 12 classes
                                DuplicateReport                                
┌──────────────────┬──────────────────────────────────────────────────────────┐
│ Metric           │ Value                                                    │
├──────────────────┼──────────────────────────────────────────────────────────┤
│ groups           │ []                                                       │
│ exact_groups     │ []                                                       │
│ similarities     │ {}                                                       │
│ hashes           │ ['11001001001001100001111000001100100101001111011011100… │
│                  │ '011010010001000100010010011010110010110001011110111110… │
│                  │ '101100001000110111001100111100100001001100000100110100… │
│                  │ '011011011010001110110111111100000011101000001101000110… │
│                  │ '011011111011110000001001111101000010101011001110010

### mining.deduplicate

Return an independent collection with one representative per duplicate group (`keep='best'`, `'quality'`, `'annotations'`, or `'first'`).

In [3]:
for policy in ("best", "quality", "annotations", "first"):
    unique = mining.deduplicate(data, keep=policy, threshold=0.95)
    print(f"keep={policy!r:<13} -> {len(unique)}/{len(data)} images retained")

keep='best'        -> 8/8 images retained
keep='quality'     -> 8/8 images retained
keep='annotations' -> 8/8 images retained
keep='first'       -> 8/8 images retained


### mining.DuplicateReport

Duplicate groups, similarities, and cached hashes returned by `find_duplicates`.

In [4]:
dup_path = duplicates.save("artifacts/tutorials/mining/duplicates_report.json")
print("Total hashed images:", duplicates.total_images)
print("Saved DuplicateReport to:", dup_path)

Total hashed images: 8
Saved DuplicateReport to: None


## Put it together

Detect duplicate groups first, inspect similarities, and produce an independent deduplicated dataset.

In [5]:
report = mining.find_duplicates(data, threshold=0.90)
cleaned = mining.deduplicate(data, keep="best", threshold=0.90)
print(f"Cleaned {len(data) - len(cleaned)} duplicate frame(s), leaving {len(cleaned)} unique images.")

Cleaned 0 duplicate frame(s), leaving 8 unique images.


## Remember

1. `find_duplicates` detects both exact (MD5 hash) and near-duplicate (pHash) images.
2. `deduplicate` retains complete samples without altering annotations.
3. Input datasets are never mutated in-place.